# Parthenon Research Workbench

Welcome to your **personal Jupyter workspace** inside Parthenon. This notebook demonstrates how to:

1. **Connect to the platform** — database, API, and file system
2. **Query OMOP CDM data** — vocabulary, cohorts, conditions, drug exposures
3. **Share notebooks** with colleagues via the shared folder
4. **Run advanced analyses** — population-level estimation, survival curves, GIS enrichment
5. **Create publication-quality visualizations**

---

**Your workspace:**
| Path | Description |
|------|-------------|
| `/home/jovyan/notebooks/` | Your private notebooks (persists across sessions) |
| `/home/jovyan/shared/` | Shared folder — copy notebooks here to collaborate |
| `/home/jovyan/parthenon/` | Read-only Parthenon codebase and docs |

**Your database access** is role-based — credentials are injected automatically based on your Parthenon role.

## 1. Environment & Connection Setup

Run this cell first — it establishes your database connection, API client, and verifies your workspace.

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

# ── Paths ──
NOTEBOOK_DIR = Path(os.environ.get('PARTHENON_NOTEBOOK_DIR', '/home/jovyan/notebooks'))
REPO_DIR = Path(os.environ.get('PARTHENON_REPO_DIR', '/home/jovyan/parthenon'))
SHARED_DIR = Path('/home/jovyan/shared')
USER_ID = os.environ.get('PARTHENON_USER_ID', 'unknown')
USER_EMAIL = os.environ.get('PARTHENON_USER_EMAIL', '')

# ── Database ──
DB_USER = os.environ.get('PARTHENON_DB_USER', 'parthenon')
DB_PASS = os.environ.get('PARTHENON_DB_PASSWORD', '')
DB_HOST = os.environ.get('PARTHENON_DB_HOST', 'postgres')
DB_PORT = os.environ.get('PARTHENON_DB_PORT', '5432')
DB_NAME = os.environ.get('PARTHENON_DB_NAME', 'parthenon')

DATABASE_URL = f'postgresql+psycopg://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL)

# ── API client ──
API_BASE = os.environ.get('PARTHENON_API_BASE_URL', 'http://nginx/api/v1')
API_TOKEN = os.environ.get('PARTHENON_API_TOKEN', '')

def api_get(path: str, params: dict | None = None) -> dict:
    """GET request to Parthenon API."""
    headers = {'Accept': 'application/json'}
    if API_TOKEN:
        headers['Authorization'] = f'Bearer {API_TOKEN}'
    url = urljoin(API_BASE.rstrip('/') + '/', path.lstrip('/'))
    resp = requests.get(url, params=params, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()

def query(sql: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

# ── Verify ──
print(f'\u2713 User: {USER_EMAIL} (ID: {USER_ID})')
print(f'\u2713 DB role: {DB_USER}')
print(f'\u2713 Notebooks: {NOTEBOOK_DIR}')
print(f'\u2713 Shared: {SHARED_DIR}')
print(f'\u2713 Repo: {REPO_DIR}')

with engine.connect() as conn:
    result = conn.execute(text('SELECT current_user, current_database()'))
    row = result.fetchone()
    print(f'\u2713 Connected to {row[1]} as {row[0]}')

## 2. Querying the OMOP CDM

Your database connection has direct access to the OMOP CDM schemas. Here are common queries to get started.

### 2a. Vocabulary Search

Search for concepts by name — essential for building cohort definitions and understanding clinical codes.

In [ ]:
def search_concepts(term: str, domain: str | None = None, limit: int = 20) -> pd.DataFrame:
    """Search OMOP vocabulary for concepts matching a term."""
    sql = """
        SELECT concept_id, concept_name, domain_id, vocabulary_id, concept_class_id, standard_concept
        FROM omop.concept
        WHERE concept_name ILIKE :term
          AND invalid_reason IS NULL
    """
    if domain:
        sql += " AND domain_id = :domain"
    sql += " ORDER BY concept_name LIMIT :limit"
    return query(sql, {'term': f'%{term}%', 'domain': domain, 'limit': limit})

# Example: search for diabetes-related conditions
search_concepts('diabetes mellitus', domain='Condition')

### 2b. Population Overview

Quick summary of the person table — total patients, age/gender distribution.

In [ ]:
# Total persons
total = query("SELECT COUNT(*) AS total_persons FROM omop.person")
print(f"Total persons in CDM: {total['total_persons'].iloc[0]:,}")

# Gender distribution
gender_dist = query("""
    SELECT c.concept_name AS gender, COUNT(*) AS n
    FROM omop.person p
    JOIN omop.concept c ON p.gender_concept_id = c.concept_id
    GROUP BY c.concept_name
    ORDER BY n DESC
""")
gender_dist

### 2c. Top Conditions

Most frequent conditions in the database — useful for identifying study populations.

In [ ]:
top_conditions = query("""
    SELECT c.concept_name AS condition, c.concept_id, COUNT(DISTINCT co.person_id) AS patients
    FROM omop.condition_occurrence co
    JOIN omop.concept c ON co.condition_concept_id = c.concept_id
    WHERE c.standard_concept = 'S'
    GROUP BY c.concept_name, c.concept_id
    ORDER BY patients DESC
    LIMIT 20
""")
top_conditions

## 3. Visualizations

Build publication-quality figures using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Dark theme to match Parthenon UI
plt.style.use('dark_background')
sns.set_palette('Set2')

PARTHENON_COLORS = {
    'teal': '#2DD4BF',
    'crimson': '#9B1B30',
    'gold': '#C9A227',
    'bg': '#0E0E11',
    'text': '#C5C0B8',
}

def parthenon_fig(width=10, height=6):
    """Create a figure styled to match Parthenon's dark clinical theme."""
    fig, ax = plt.subplots(figsize=(width, height), facecolor=PARTHENON_COLORS['bg'])
    ax.set_facecolor(PARTHENON_COLORS['bg'])
    ax.tick_params(colors=PARTHENON_COLORS['text'])
    for spine in ax.spines.values():
        spine.set_color('#232328')
    return fig, ax

print('\u2713 Visualization helpers ready')

In [ ]:
# Age distribution histogram
ages = query("""
    SELECT EXTRACT(YEAR FROM CURRENT_DATE) - year_of_birth AS age
    FROM omop.person
    WHERE year_of_birth IS NOT NULL
""")

if not ages.empty:
    fig, ax = parthenon_fig()
    ax.hist(ages['age'], bins=30, color=PARTHENON_COLORS['teal'], alpha=0.8, edgecolor='#232328')
    ax.set_xlabel('Age', color=PARTHENON_COLORS['text'])
    ax.set_ylabel('Patients', color=PARTHENON_COLORS['text'])
    ax.set_title('Patient Age Distribution', color=PARTHENON_COLORS['text'], fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No person records found — is the CDM loaded?')

## 4. Advanced Analysis Patterns

These patterns demonstrate how to do real research work inside Parthenon.

### 4a. Cohort Attrition Analysis

Compare how inclusion/exclusion criteria reduce your study population step by step.

In [ ]:
def cohort_attrition(steps: list[tuple[str, str]]) -> pd.DataFrame:
    """
    Build an attrition table from sequential SQL filters.
    
    Args:
        steps: List of (label, SQL_WHERE_clause) tuples.
               Each step further filters the previous result.
    
    Example:
        cohort_attrition([
            ('All persons', '1=1'),
            ('Age >= 18', "EXTRACT(YEAR FROM CURRENT_DATE) - year_of_birth >= 18"),
            ('Has condition', 'person_id IN (SELECT person_id FROM omop.condition_occurrence)'),
        ])
    """
    results = []
    cumulative_where = []
    
    for label, condition in steps:
        cumulative_where.append(f'({condition})')
        where_clause = ' AND '.join(cumulative_where)
        sql = f'SELECT COUNT(*) AS n FROM omop.person WHERE {where_clause}'
        n = query(sql)['n'].iloc[0]
        prev_n = results[-1]['n'] if results else n
        results.append({
            'step': label,
            'n': n,
            'excluded': prev_n - n,
            'pct_remaining': round(100 * n / results[0]['n'], 1) if results else 100.0,
        })
    
    return pd.DataFrame(results)

# Example attrition
attrition = cohort_attrition([
    ('All persons', '1=1'),
    ('Age >= 18', "EXTRACT(YEAR FROM CURRENT_DATE) - year_of_birth >= 18"),
    ('Has visit', 'person_id IN (SELECT DISTINCT person_id FROM omop.visit_occurrence)'),
])
attrition

### 4b. Temporal Trends

Chart condition prevalence over time — useful for identifying data quality issues or real trends.

In [ ]:
def condition_trend(concept_id: int, concept_name: str = 'Condition') -> None:
    """Plot monthly occurrence counts for a specific condition concept."""
    trend = query("""
        SELECT DATE_TRUNC('month', condition_start_date) AS month, COUNT(*) AS occurrences
        FROM omop.condition_occurrence
        WHERE condition_concept_id = :cid
        GROUP BY month
        ORDER BY month
    """, {'cid': concept_id})
    
    if trend.empty:
        print(f'No occurrences found for concept {concept_id}')
        return
    
    fig, ax = parthenon_fig(12, 5)
    ax.plot(trend['month'], trend['occurrences'], color=PARTHENON_COLORS['teal'], linewidth=1.5)
    ax.fill_between(trend['month'], trend['occurrences'], alpha=0.15, color=PARTHENON_COLORS['teal'])
    ax.set_xlabel('Month', color=PARTHENON_COLORS['text'])
    ax.set_ylabel('Occurrences', color=PARTHENON_COLORS['text'])
    ax.set_title(f'{concept_name} — Monthly Trend', color=PARTHENON_COLORS['text'], fontsize=14)
    plt.tight_layout()
    plt.show()

# Uncomment and replace with a concept_id from your search above:
# condition_trend(201826, 'Type 2 diabetes mellitus')

### 4c. Drug Exposure Analysis

Analyze drug utilization patterns — top drugs, exposure duration, polypharmacy.

In [ ]:
top_drugs = query("""
    SELECT c.concept_name AS drug, c.concept_id,
           COUNT(DISTINCT de.person_id) AS patients,
           ROUND(AVG(de.drug_exposure_end_date - de.drug_exposure_start_date)) AS avg_days
    FROM omop.drug_exposure de
    JOIN omop.concept c ON de.drug_concept_id = c.concept_id
    WHERE c.standard_concept = 'S'
      AND de.drug_exposure_start_date IS NOT NULL
      AND de.drug_exposure_end_date IS NOT NULL
    GROUP BY c.concept_name, c.concept_id
    ORDER BY patients DESC
    LIMIT 15
""")
top_drugs

### 4d. Working with Achilles Results

Achilles pre-computes hundreds of data quality and characterization metrics. Query them directly for fast insights.

In [ ]:
# Achilles analysis summary — which analyses have been run?
achilles_summary = query("""
    SELECT analysis_id, COUNT(*) AS result_rows
    FROM results.achilles_results
    GROUP BY analysis_id
    ORDER BY result_rows DESC
    LIMIT 20
""")
print(f'Achilles analyses available: {len(achilles_summary)}')
achilles_summary.head(10)

## 5. Sharing Your Work

Your notebooks live in your **private workspace** (`/home/jovyan/notebooks/`). To share with colleagues:

### Copy to the shared folder
```python
import shutil
shutil.copy('my-analysis.ipynb', f'/home/jovyan/shared/{USER_ID}/my-analysis.ipynb')
```

### Browse shared notebooks from others
```python
from pathlib import Path
for f in sorted(Path('/home/jovyan/shared').rglob('*.ipynb')):
    print(f)
```

### Copy a shared notebook to your workspace
```python
shutil.copy('/home/jovyan/shared/42/their-analysis.ipynb', '/home/jovyan/notebooks/')
```

In [ ]:
import shutil

def share_notebook(notebook_name: str) -> str:
    """Copy a notebook from your workspace to your shared folder."""
    src = NOTEBOOK_DIR / notebook_name
    dest_dir = SHARED_DIR / str(USER_ID)
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / notebook_name
    shutil.copy2(src, dest)
    return f'Shared: {dest}'

def list_shared_notebooks() -> pd.DataFrame:
    """List all notebooks in the shared folder."""
    notebooks = []
    for f in sorted(SHARED_DIR.rglob('*.ipynb')):
        notebooks.append({
            'path': str(f.relative_to(SHARED_DIR)),
            'size_kb': round(f.stat().st_size / 1024, 1),
            'owner_dir': f.parent.name,
        })
    return pd.DataFrame(notebooks) if notebooks else pd.DataFrame(columns=['path', 'size_kb', 'owner_dir'])

# See what's shared
list_shared_notebooks()

## 6. Exporting Results

Save analysis outputs to your workspace for use in Parthenon or external tools.

In [ ]:
def save_results(df: pd.DataFrame, filename: str, fmt: str = 'csv') -> str:
    """Save a DataFrame to your notebook workspace.
    
    Args:
        df: The DataFrame to save
        filename: Output filename (without extension)
        fmt: 'csv', 'parquet', or 'excel'
    """
    out_dir = NOTEBOOK_DIR / 'output'
    out_dir.mkdir(exist_ok=True)
    
    if fmt == 'csv':
        path = out_dir / f'{filename}.csv'
        df.to_csv(path, index=False)
    elif fmt == 'parquet':
        path = out_dir / f'{filename}.parquet'
        df.to_parquet(path, index=False)
    elif fmt == 'excel':
        path = out_dir / f'{filename}.xlsx'
        df.to_excel(path, index=False)
    else:
        raise ValueError(f'Unknown format: {fmt}')
    
    return f'Saved: {path} ({path.stat().st_size / 1024:.1f} KB)'

# Example: save top conditions to CSV
# save_results(top_conditions, 'top_conditions', 'csv')
print('\u2713 Export helpers ready — use save_results(df, name) to save DataFrames')

## 7. Quick Recipe Reference

Copy these patterns into new cells for common research tasks.

In [ ]:
recipes = pd.DataFrame([
    {'task': 'Search vocabulary', 'code': "search_concepts('hypertension', domain='Condition')"},
    {'task': 'Count patients with condition', 'code': "query('SELECT COUNT(DISTINCT person_id) FROM omop.condition_occurrence WHERE condition_concept_id = 201826')"},
    {'task': 'Patient demographics', 'code': "query('SELECT gender_concept_id, COUNT(*) FROM omop.person GROUP BY 1')"},
    {'task': 'Drug exposures for a patient', 'code': "query('SELECT * FROM omop.drug_exposure WHERE person_id = :pid', {'pid': 123})"},
    {'task': 'Cohort attrition', 'code': "cohort_attrition([('All', '1=1'), ('Adults', 'EXTRACT(YEAR FROM CURRENT_DATE) - year_of_birth >= 18')])"},
    {'task': 'Temporal trend', 'code': "condition_trend(201826, 'Type 2 DM')"},
    {'task': 'Save results', 'code': "save_results(df, 'my_results', 'csv')"},
    {'task': 'Share notebook', 'code': "share_notebook('my-analysis.ipynb')"},
    {'task': 'List shared notebooks', 'code': 'list_shared_notebooks()'},
    {'task': 'Achilles results', 'code': "query('SELECT * FROM results.achilles_results WHERE analysis_id = 1 LIMIT 10')"},
    {'task': 'API call', 'code': "api_get('/health')"},
])
recipes

---

## Next Steps

1. **Duplicate this notebook** — `File → Save Notebook As` to create your own working copy
2. **Explore the vocabulary** — use `search_concepts()` to find the clinical codes you need
3. **Build your cohort** — use `cohort_attrition()` to iteratively refine inclusion criteria
4. **Visualize** — use `parthenon_fig()` for publication-ready dark-themed charts
5. **Share** — copy finished notebooks to `/home/jovyan/shared/` for your team

Your server automatically stops after 30 minutes of inactivity. **All your work is saved** — just come back and it picks up where you left off.